# Creating the XGBoost 
This notebook reveals the need for calibration

--- 
# Load Modules/Libraries

---

In [1]:
from importlib.metadata import version

from pathlib import Path
from typing import Union 

from dotenv import load_dotenv
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, brier_score_loss, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import seaborn as sns
import xgboost as xgb


In [2]:
pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "pandas",
        "xgboost"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.8
numpy version: 2.4.4
pandas version: 3.0.2
xgboost version: 3.2.0


--- 
# Retreieving the Data

---

In [3]:
# Data File constants
file_name = "cleaned_data.csv"
data_file_path = Path("data") / file_name

df = pd.read_csv(data_file_path)

--- 
# Get Training, Testing, and Validation Test Sets

---

##### Perform stratified splitting
Stratified splitting is crucial for small (< 1000 observations), imbalanced datasets. It ensures the target distribution (and implicitly the rare feature distributions) remains the same in both sets.

Using `stratify=y` in the train_test_split function ensures that your model evaluation is based on a representative sample of your data.


##### Encoding and Scaling
After splitting, encode categorical variables and scale continuous ones to prevent data leakage. In machine learning, if you encode or scale your entire dataset before splitting it into training and testing sets, the "testing" information "leaks" into the "training" process.

In [4]:
X = df.drop('target', axis=1)
y = df['target']

# 1. First Split: Separate out the Test set (20% of total)
# This leaves 80% for Train + Validation
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# 2. Second Split: Split the remaining 80% into Train and Validation
# To get a 60/20/20 split, we take 25% of the 80% (0.25 * 0.80 = 0.20)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, 
    test_size=0.25, 
    random_state=42, 
    stratify=y_train_val
)


#### Encoding the Key Categorical Features


| Feature | Type | Action Required |
| :--- | :--- | :--- |
| **cp** | Categorical (Nominal) | Dummy/One-Hot Encoding |
| **restecg** | Categorical (Nominal) | Dummy/One-Hot Encoding |
| **slope** | Categorical (Ordinal) | Dummy/One-Hot Encoding |
| **thal** | Categorical (Nominal) | Dummy/One-Hot Encoding |
| **ca** | Categorical (Discrete) | Dummy/One-Hot Encoding (Commonly) |
| **sex, fbs, exang** | Binary | None (Keep as 0/1) |


##### The specific features that require dummy variables (or one-hot encoding) are:
* **cp** (Chest Pain Type): Usually has 4 levels (0–3) representing typical angina, atypical angina, non-anginal pain, and asymptomatic.

* **restecg** (Resting Electrocardiographic Results): Typically has 3 levels (0, 1, 2) indicating normal, ST-T wave abnormality, or left ventricular hypertrophy.

* **slope** (The Slope of the Peak Exercise ST Segment): Usually has 3 levels (0, 1, 2) for upsloping, flat, and downsloping.

* **thal** (Thalassemia): Features 3 levels (often coded as 3, 6, 7 or 1, 2, 3) representing normal, fixed defect, and reversible defect.

* **ca** (Number of Major Vessels): While technically a count (0-4), it is often treated as a categorical variable in this specific classification task to improve model performance.

In [5]:
# --- PREVENTING LEAKAGE DURING ENCODING ---

# 3. Specify Categorical and Numerical Features
# We use 'columns' to specify categorical features. 
# 'drop_first=True' avoids the dummy variable trap.
categorical_cols = ["ca", "cp", "exang", "fbs", "restecg", "sex", "slope", "thal"]
numerical_cols = ["age", "chol", "oldpeak", "thalach", "trestbps"]

### Preprocessing and Creating a Pipeline

In [6]:
# Perform the Preprocessing (Scaling and Encodings)
# ColumnTransformer handles OneHotEncoder for categorical data and StandardScaler for numerical data

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Creat base XGBoost Classifier model which will be calibrated later
base_xgb = xgb.XGBClassifier(eval_metric="logloss", random_state=42)
calibrated_xgb = CalibratedClassifierCV(estimator=base_xgb, method="sigmoid", cv=3)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("clf", base_xgb)])

---

### Perform the GridSearch with Nested Parameters
GridSearchCV tunes nested XGBoost hyperparameters within the Pipeline.

When you run `grid_search.fit(X_train, y_train)`, scikit-learn does two things:

* Search: It tests every combination of parameters in your `param_grid` using cross-validation to find which one performs best.

* Refit: Once the best parameters are found, it automatically re-trains the pipeline on the entire training set (`X_train`) using those specific settings.


#### Remember with Scaling Feature Values - make sure to prevent leakage
When you use a `StandardScaler`, it calculates the mean and standard deviation of the data to shift everything to a center of 0.

* **The Leak**: If you scale the whole dataset first, the scaler uses the values from the test set to calculate that mean.

* **The Consequence**: Your training data now "knows" the average value of the test data. The model is no longer being tested on truly "unseen" data because the test set's distribution influenced the training set's values.

* **The Fix**: Always fit the scaler on the Train set only, then transform both the Train and Test sets using those training parameters.

In [12]:
# Hyperparameter tuning using Training Data
param_grid = {
    "clf__max_depth": [3, 5],
    "clf__learning_rate": [0.01, 0.1],
    "clf__n_estimators": [50, 100]
}

#n_jobs = -1 says use all available processors
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="roc_auc", refit=True)

#  --- MAKE SURE TO PREVENT LEAKAGE DURING SCALING  using only the Training data---
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__learning_rate': [0.01, 0.1], 'clf__max_depth': [3, 5], 'clf__n_estimators': [50, 100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is display

### Calibrate the base XGBoost model : Perform probability calibration (using separate Validation data)
Calibration is used to ensure predicted probabilities match real-world frequencies.

Used strictly for Calibration. Since XGBoost's raw probabilities can be biased, `CalibratedClassifierCV` needs "fresh" data that wasn't used for the initial training to properly map scores to actual probabilities. Hence, the use of the validation training set.

Here, `cv="prefit"` because the pipeline was already trained during `GridSearch`

In [14]:
# grid_search.best_estimator_ is the final, optimized version of your entire machine learning Pipeline
# best_estimator_ is a Pipeline object containing: 
# * The Preprocessor: Your ColumnTransformer (StandardScaler + OneHotEncoder)
# * The Model: The XGBClassifier with the specific max_depth, learning_rate, and n_estimators that scored the highest.
best_pipeline = grid_search.best_estimator_

#The calibrated XGBoost model.
calibrated_model = CalibratedClassifierCV(
    estimator=best_pipeline, 
    method='sigmoid', 
    cv=5
)
calibrated_model.fit(X_val, y_val)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2","Pipeline(step...=None, ...))])"
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the col

### Final Evaluation Using the X_test Data

The `X_test` data was held out entirely until the very end to give an honest assessment of how the calibrated model performs on data it has never seen. 

In [16]:
y_probs = calibrated_model.predict_proba(X_test)[:, 1]
y_preds = calibrated_model.predict(X_test)

In [17]:
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}")
print(f"Test Brier Score: {brier_score_loss(y_test, y_probs):.4f}")
print("\nFinal Test Classification Report:")
print(classification_report(y_test, y_preds))

Best Parameters: {'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100}
Test ROC-AUC: 0.8831
Test Brier Score: 0.1596

Final Test Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.68      0.72        28
           1       0.75      0.82      0.78        33

    accuracy                           0.75        61
   macro avg       0.76      0.75      0.75        61
weighted avg       0.75      0.75      0.75        61



---
# Training the Models Using Some Non Neural Network Models and Getting the Probabilities

---

In [ ]:
# Classifier (clf) Models used
clf_models = {
    "nb": GaussianNB, #naive bayes classifier
    "rf": RandomForestClassifier,
    "svc": SVC #support vector classifier 
}

In [ ]:
model_to_probs = {}
model_name_to_trained_model = {}
brier_scores = {} 

for model_name, model in clf_models.items():
    print(model)

    # Add parameters specific to each classifier.
    if model == SVC:
        clf = model(probability=True)
    elif model == LogisticRegression:
        clf = model(solver='liblinear')
    else:
        clf = model()

    # Fit the model to the scaled data
    clf.fit(X_train_scaled, y_train)

    # Get predicted probabilities for the positive class (1)
    prediction_probs_train = clf.predict_proba(X_train_scaled)[:,1]
    prediction_probs_test = clf.predict_proba(X_test_scaled)[:,1]
    prediction_probs_val = clf.predict_proba(X_val_scaled)[:,1]

    # CALCULATE BRIER SCORES - the Mean Squared Error (MSE) applied to predicted probabilities
    # Measures how well a model is calibrated
    # We compare the predicted probabilities against the true labels (y)
    bs_train = brier_score_loss(y_train, prediction_probs_train)
    bs_test = brier_score_loss(y_test, prediction_probs_test)
    bs_val = brier_score_loss(y_val, prediction_probs_val)

    # Store for later
    brier_scores[model_name] = {'train': bs_train, 'test': bs_test, 'valid': bs_val}
    print(f"--- {model_name} Brier Scores ---")
    print(f"Train: {bs_train:.4f} | Test: {bs_test:.4f} | Valid: {bs_val:.4f}\n")

    model_to_probs[model_name] = {
        'train': prediction_probs_train, 
        'test': prediction_probs_test, 
        'valid': prediction_probs_val
    }

    plt.figure(figsize=(20,4))
    
    plt.subplot(1,2,1)
    sns.histplot(prediction_probs_train)
    plt.title(f"{model_name} - train", fontsize=20)
    
    plt.subplot(1,2,2)
    sns.histplot(prediction_probs_test)
    plt.title(f"{model_name} - test", fontsize=20)
    
    model_name_to_trained_model[model_name] = clf

---
# Plot Predicted Probabilities vs Empirical Probabilities (also Known as Realiability Diagrams)

We are plotting "Probabilities vs Empirical Probabilities" because we want to see how much we can trust the "confidence" of our model.

In classification, a model doesn't just give a label; it gives a probability (e.g., 0.8 or 80%). Calibration analysis checks if that 80% actually reflects reality.

* **Predicted Probability**: What the model thinks is the chance of an event (e.g., "There is a 70% chance this patient has heart disease").

* Empirical Probability: What actually happens in the real world for cases with that prediction (e.g., "Out of 100 people the model gave a 70% score to, did exactly 70 of them actually have heart disease?").

#### Why It Matters
* Reliability: If a weather app says there’s a 90% chance of rain, you expect it to rain almost every time. If it only rains 40% of the time when it says 90%, the model is overconfident.

* Decision Making: In medical or financial models, a "0.51" prediction is very different from a "0.99" prediction. If the model is poorly calibrated, you can't use these scores to prioritize high-risk patients.

* Model Comparison: Two models might have the same Accuracy, but one might be much better calibrated (more "honest" about its uncertainty) than the other.

#### How to Read the Plot (Calibration Curve)
* The Diagonal Line (y = x): This represents a perfectly calibrated model.

* Below the Diagonal: The model is overconfident. It predicts high probabilities, but the actual events happen less often.

* Above the Diagonal: The model is underconfident. It predicts low probabilities, but the events happen more often than it thinks




### ---`Warning`--- Binning Process of a Small Data Set and Quantile vs Uniform

#### Bin sizes are important
When plotting calibration curves for small datasets, we have to cognizant of the effects of binning of the data on the plots with 

`calibration_curve()`


With only ~300 total observations (~180 in the test set and ~60 in the validation test set), if we don't find the sweet spot of the `bin` size as each point on the figure would attempt to represent that number of observations/bin size. (e.g., bins=10 means each point on the figure represents ~ 18 observations 180/10).

**The problem**: Overly large bins result in the `"Zero or Hero"` effect where in the figure the curves can suddely drop to 0.0. This effect happens because there are only a few observations in a specific probability range. 

At the same time, this small dataset can experience high variance because each bin point consists of only a few observations. The plot essentially will show more statistical noise and not necessarily the Machine Learning model's performance trend. 

#### The Fix reduce the bin size. 
This reduction puts more samples in each bin, which will smooth out those wild swings.




---

### ---`Warning`--- Quantile vs Uniform Strategy Usage

The default `strategy=uniform` in `calibration_curve()`

**The Problem**: strategy='uniform' creates bins of equal width (0.0–0.1, 0.1–0.2, etc.). If a bin has only two observations in it, the graph swings wildly.

#####  The Fix use `Quantile`
Quantile ensures that every dot on the graph represents the same number of observations. This "averages out" the noise and gives a much more honest view of the model's performance.


In [ ]:
# Calculate true (Y-axis) vs. predicted probabilities (X-axis) for the bins
# n_bins=10 is standard; higher requires more data
   
for model_name, pred_prob_dict in model_to_probs.items():
    
    prob_emp, prob_pred = calibration_curve(y_test, pred_prob_dict["test"], n_bins=5, strategy="quantile",)

    plt.figure(figsize=(10, 4))
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated', color='gray') # Perfect baseline
    plt.plot(prob_pred, prob_emp, marker='.', label=model_name)
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives (Empirical)')
    plt.title('Calibration Curve / Reliability Diagram')
    plt.legend()
    plt.grid(True)
    plt.show()


### Commentary on the Empirical vs Predicted Plots of NB, RL, and SV models

* **The Naive Bayes (nb)** The blue line representing the nb model shows significant miscalibration, characterized by an "Inverted S-shape" relative to the perfectly calibrated dashed line:

    * `Low-End Under-confidence`: At predicted probabilities near 0.08, the actual fraction of positives is much higher (roughly 0.25). This means when the model is "sure" something is unlikely, it actually happens more often than predicted.

    * `High-End Over-confidence`: At predicted probabilities near 0.80, the actual fraction of positives is lower (roughly 0.67). Here, the model is "too bold"—it predicts an 80% chance of a heart issue, but it only occurs about 67% of the time.

    * `Central Crossing`: The model is most accurate near the 0.5 probability mark, where it crosses the dashed line.


* **The Random Forest (rf)** The blue line reliability curve shows a classic S-shape, which shows systemic miscalibration. The RF model is too cautious.

* Overconfidence in the Low/Mid-Range: Between predicted probabilities of 0.2 and 0.6, the blue line is below the dashed diagonal. This means the model is predicting, for example, a 60% risk, but the actual fraction of heart disease cases is only about 50%.

* Underconfidence in the High-Range: Above 0.75, the line crosses above the diagonal. When the model predicts an 80% risk, the actual frequency is closer to 85%. It is under-estimating the risk when it gets more certain.

* Why this might be happening?
    * Because an RF is an ensemble (an average) of many individual decision trees:
        * The averaging process naturally pulls predictions away from the extreme ends (0 and 1) and pushes them toward the center.
        * Individual trees might predict 0 or 1, but the forest's average rarely reaches those extremes unless every tree agrees perfectly. This results in the "sigmoidal" or S-shaped curve.


* **The Support Vector Machine Classifier (svc)** For almost the entire range (specifically from 0.1 to 0.6 predicted probability), the blue line sits below the dashed diagonal meaning the model is overconfident. The model becomes most accurate around the 0.6 to 0.8 range, where it hugs the diagonal line closely. In this specific "medium-high risk" zone, the SVC is actually very well-calibrated naturally. At the very top of the scale (predicted probability 0.9), the curve shoots back up to the diagonal meaning that is underconfident.

    * if you used this uncalibrated SVC in a hospital:
        * Low-risk patients would be told they have a moderate risk (e.g., 30%), leading to unnecessary stress or follow-up tests.
        * High-risk patients (above 70%) would receive fairly accurate assessments.

    * Why this might be happening?
        * Because Support Vector Machines are "maximum margin" classifiers. They are designed to find the best boundary to separate classes, not to produce accurate probabilities. 
        * Because the SVC focuses on the points closest to the decision boundary (the support vectors), the "scores" it produces are distances from that boundary.


### In short, all the models need to be calibarated to make them more reliable!

---

# Calibrating the Models

---

### sckikit-learn's CalibratedClassififerCV

In scikit-learn, the `CalibratedClassifierCV` class is the primary tool for this. It supports two main methods: 

* **Platt Scaling (method='sigmoid')**: Fits a Logistic Regression model to the scores. It works best when the calibration error follows a sigmoidal (S-shaped) pattern.

* **Isotonic Regression (method='isotonic')**: A non-parametric approach that fits a piecewise-constant, non-decreasing function. It is more flexible but requires more data to avoid overfitting


### Matching the Calibration technique to the clf models: nb,rf, and svc

Choosing the "best" calibration technique depends more on your dataset size and the shape of the miscalibration than just the model type itself. However, there are industry-standard pairings for Random Forest (RF) and Support Vector Classifiers (SVC).

* **The Naive Bayes (nb)** model: If we had a larger dataset (>1000 observations) we would consider `Isontonic Regression`, but here we will use `Platt Scaling technique`. Platt Scaling only needs to find two numbers (the A and B parameters of a sigmoid function). It is very hard for the model to "hallucinate" patterns with only two variables.


* **The Random Forest (rf)** model: Random Forests tend to produce "bunched" probabilities. Because they average many trees, it is rare for them to predict exactly 0 or 1, often leading to a complex, non-linear miscalibration that isn't perfectly sigmoidal. Regression is non-parametric. It doesn't assume a specific shape (like a sigmoid); instead, it fits a piece-wise constant non-decreasing function to your data. This allows it to correct any "wiggly" or complex monotonic distortions that a simple sigmoid might miss. We see the RF graph is very wiggly. The general convention here would be to us `Isontonic Regression` which non-parametric meaning it does not assume a spcific shape like a sigmoid. However, it is known to work better for datasets of greater than 1000 observations because it is very prone to overfitting. If your calibration dataset is small, `Isotonic Regression` will "memorize" the noise (creating a jagged line like the SVC graph), which actually makes your model worse. As a result, to calibrate the `rf` model I will use the `Platt Scaling technique`.

* **The Support Vector Machine Classifier (svc)** model: Support Vector Machines are not "probabilistic" by nature—they care about the distance from the decision boundary, not the likelihood of a class. To get these probabilities, tools like Scikit-Learn usually use Platt Scaling (fitting a logistic regression on top of the SVM outputs). Platt Scaling is parametric, meaning it only needs to learn two parameters (A and B). This makes it much more stable than other methods when you have a smaller dataset (typically under 1,000 samples) like what we have ~ 300 observations.



#### Summary of Calibrators

| Model | Justification | Calibration Technique|
| :--- | :--- | :--- |
| **nb** | under 1000 datapoints, avoid overfitting with Isontonic Regression| Platt Scaling |
| **rf** | under 1000 datapoints, avoid overfitting with Isontonic Regression | Platt Scaling |
| **svc** | under 1000 datapoints and typical match-up with SVM models | Platt Scaling |


In [ ]:
# Calibrated_Classifier (clf) Models used
calibrated_clf_models = {
    "nb_calibrated": GaussianNB(), #naive bayes classifier
    "rf_calibrated": RandomForestClassifier(n_estimators=100, random_state=42),
    "svc_calibrated": SVC(kernel="linear") #support vector classifier 
}

In [ ]:
cal_model_to_probs = {}
cal_model_name_to_trained_model = {}


for model_name, model in calibrated_clf_models.items():
    print(model)

    clf = CalibratedClassifierCV(model, method='sigmoid', cv=5)

    # Fit the model to the scaled data
    clf.fit(X_train_scaled, y_train)

    # Get predicted probabilities for the positive class (1)
    prediction_probs_train = clf.predict_proba(X_train_scaled)[:,1]
    prediction_probs_test = clf.predict_proba(X_test_scaled)[:,1]
    prediction_probs_val = clf.predict_proba(X_val_scaled)[:,1]

    cal_model_to_probs[model_name] = {
        'train': prediction_probs_train, 
        'test': prediction_probs_test, 
        'valid': prediction_probs_val
    }

    # CALCULATE BRIER SCORES - the Mean Squared Error (MSE) applied to predicted probabilities
    # Measures how well a model is calibrated
    # We compare the predicted probabilities against the true labels (y)
    bs_train = brier_score_loss(y_train, prediction_probs_train)
    bs_test = brier_score_loss(y_test, prediction_probs_test)
    bs_val = brier_score_loss(y_val, prediction_probs_val)

    # Store for later
    brier_scores[model_name] = {'train': bs_train, 'test': bs_test, 'valid': bs_val}
    print(f"--- {model_name} Brier Scores ---")
    print(f"Train: {bs_train:.4f} | Test: {bs_test:.4f} | Valid: {bs_val:.4f}\n")


    plt.figure(figsize=(20,4))
    
    plt.subplot(1,2,1)
    sns.histplot(prediction_probs_train)
    plt.title(f"{model_name} - train", fontsize=20)
    
    plt.subplot(1,2,2)
    sns.histplot(prediction_probs_test)
    plt.title(f"{model_name} - test", fontsize=20)
    
    cal_model_name_to_trained_model[model_name] = clf

In [ ]:
# Calculate true (Y-axis) vs. predicted probabilities (X-axis) for the bins
# n_bins=10 is standard; higher requires more data
   
for model_name, pred_prob_dict in cal_model_to_probs.items():
    
    prob_emp, prob_pred = calibration_curve(y_test, pred_prob_dict["test"], n_bins=5, strategy="quantile")

    plt.figure(figsize=(10, 4))
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated', color='gray') # Perfect baseline
    plt.plot(prob_pred, prob_emp, marker='.', label=model_name)
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives (Empirical)')
    plt.title('Calibration Curve / Reliability Diagram')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
print(brier_scores)

### Commentary on the Calibrated Models

##### Brier Summary - Measures how well a model is calibrated

All models improved with calibration as the Brier scores show a decrease!

| Model | Brier Score Before | Brier Score After Calibration Technique|
| :--- | :--- | :--- |
| **nb** | 0.127 | 0.120  |
| **rf** | 0.135 | 0.132 |
| **svc** |0.119 | 0.116 |


* **The Naive Bayes (nb)** Yes, calibration improves the utility of the nb model. 

    * **Before**: The model was likely over-confident at the high end and under-confident at the low end.

    * **After**: The blue line now tracks the diagonal much more closely, especially in the 0.25 to 0.80 range. This means that when the model predicts a 60% chance of heart disease, the real-world frequency is now actually close to 60%.

* **The Random Forest (rf)** Yes, it improved the model, but at a price
    * The model is now much more "honest." If you look at the overall distance from the dashed line, the calibrated version is clearly closer to the diagonal than the original S-shaped curve. 

    * The Trade-off: The "staircase" effect means the model has lost some resolution. For example, a patient with a 40% predicted risk and a patient with a 55% predicted risk are now essentially treated as having the same "actual" risk (about 42%).


* **The Support Vector Machine Classifier (svc)** There is an improvement. 

    * **Reliability improved --Reliability Gain**: The model is objectively better. The area between the blue line and the dashed line is much smaller now than in the uncalibrated version. This means the Brier Score improved (decreased). A doctor can trust that a "90%" prediction means the patient almost certainly has a condition. However, between 40% and 60%, the doctor should know the model is still slightly "exaggerating" the risk.

    * **Correction of the Sag**: Standard SVCs usually have a deep curve below the diagonal (over-confidence). Calibration has pulled this blue line significantly closer to the center.

    * **The Over/Under Pivot**: Notice where the blue line crosses the dashed line (around 0.75 on the x-axis).

        * **Below 0.75**: The model is still slightly over-confident. For example, when it predicts a 60% (0.6) probability, the actual frequency of heart disease is only 50% (0.5).

        * **Above 0.75**: The model shifts to being under-confident. At the far right, it predicts roughly 92%, but the actual outcomes are at 100%. In a medical context, this is a "safe" error because the the model is slightly more cautious than it needs to be with high-risk patients.




# How to Use the Calibration Models In Practice

Now, that it can be seen that calibrating the models help to make models more accurate at classification (i.e., prediction). A natural questions is how to use these models effectively.

The hard work of ensuring the models are "honest" (calibrated) has been compleded, using them in practice is where the real value comes in, especially for a medical dataset like hearts.csv.

This section provides an example using only one of the calibrated models.

### Save the Calibrated Model

So we don't have to re-train and re-calibrate every time. 

Use `joblib` to save the entire calibrated object (which includes both the base model and the calibration scaling).


In [ ]:

# Save the calibrated SVC or RF model
joblib.dump(cal_model_name_to_trained_model["svc_calibrated"], 'calibrated_heart_model.pkl')


### Predict Probabilities (Not Just Labels)

In a heart disease context, a simple "0" or "1" (No Heart Disease/ Heart Disease) isn't enough. You want the calibrated probability. Use `.predict_proba()` to get the percentage risk.

In [ ]:
# Load the model
loaded_model = joblib.load('calibrated_heart_model.pkl')

# Let's say this is a new patient's data (age, sex, cp, trestbps, chol, etc.)
# Recall to test a heart disease classification model with new patient data after 
# it has been scaled and encoded, you must apply the exact same transformation parameters 
# (mean, standard deviation, and category mappings) used during the training phase.
# For simplicity, I used the first row of the X_test_scaled data set
# An alternate would have been to apply a ColumnTransformer to apply scaling and one-hot encoding
new_patient = X_test_scaled[1,:]

# Reshape the data
# Scikit-learn expects a 2D array (rows, columns). 
# .reshape(1, -1) turns our list into a single row.
new_patient_reshaped = new_patient.reshape(1, -1)

# Scale the data
# Use the SAME scaler object you used for X_train. 
# Use .transform(), NOT .fit_transform()!
new_patient_scaled = scaler.transform(new_patient_reshaped)

# STEP 3: Predict with the Calibrated Model
prob_sick = cal_model_name_to_trained_model["svc_calibrated"].predict_proba(new_patient_scaled)[:, 1]

print(f"Final Calibrated Result: {prob_sick[0]*100:.1f}% chance of heart disease.")


### Setting a "Clinical" Decision Threshold
In standard classification, the threshold is 0.5. But in heart disease, a False Negative (missing a sick patient) is much more dangerous than a False Positive (running an extra test on a healthy person).

**Without calibration, a "0.30" score might actually mean a 10% or 50% risk. With your calibrated model, you know 0.30 means approximately 30% frequency in the real world.**

Because your model is now calibrated, you can set a "Safety Threshold" with confidence:

* High Sensitivity (Safe): If risk > 0.30, refer for further testing.
* Standard: If risk > 0.50, diagnose as positive.
* High Specificity (Conservative): If risk > 0.80, proceed to immediate treatment.



#### Patient Risk Stratification
You can use the calibrated output to sort patients into "Risk Tiers" for a hospital dashboard:


| Risk Score | Category | Action |
| :--- | :--- | :--- |
| 0.0 - 0.2 | Low Risk | Routine check-up in 12 months. |
| 0.2 - 0.5 | Elevated | Monitor and suggest lifestyle changes. |
| 0.5 - 0.8 | High Risk | Schedule diagnostic imaging (ECG/Stress Test). |
| 0.8 - 1.0 | Critical | Urgent specialist consultation. |
